In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter
from sklearn.preprocessing import LabelEncoder


In [7]:
# Load dataset from a CSV file
# Replace 'path_to_your_csv_file.csv' with the actual path to your CSV file
data = pd.read_csv("/Users/vishrutgrover/coding/pokerai/handreco/poker_dataset_processed.csv")


In [8]:
def is_straight(ranks):
    """Check if the hand is a straight."""
    if len(set(ranks)) != 5:
        return False
    return max(ranks) - min(ranks) == 4

def is_flush(suits):
    """Check if the hand is a flush."""
    return len(set(suits)) == 1

def hand_rank(suits, ranks):
    """Determine the rank of a poker hand."""
    count = Counter(ranks)
    unique_ranks = set(ranks)
    sorted_ranks = sorted(unique_ranks, reverse=True)

    if is_straight(ranks) and is_flush(suits):
        return 'Straight Flush'
    elif 4 in count.values():
        return 'Four of a Kind'
    elif sorted(count.values(), reverse=True) == [3, 2]:
        return 'Full House'
    elif is_flush(suits):
        return 'Flush'
    elif is_straight(ranks):
        return 'Straight'
    elif 3 in count.values():
        return 'Three of a Kind'
    elif list(count.values()).count(2) == 2:
        return 'Two Pair'
    elif 2 in count.values():
        return 'One Pair'
    else:
        return 'High Card'

def evaluate_hand(row):
    suits = [row[f'S{i}'] for i in range(1, 8)]
    ranks = [row[f'C{i}'] for i in range(1, 8)]
    return hand_rank(suits, ranks)

In [10]:
# Apply the hand evaluation function to create the 'HandRank' column
data['HandRank'] = data.apply(evaluate_hand, axis=1)

# Encode the hand ranks into numerical labels
label_encoder = LabelEncoder()
data['HandRank'] = label_encoder.fit_transform(data['HandRank'])


# Split data into features and labels
X = data.iloc[:, :-1].values  # All columns except 'HandRank'
y = data['HandRank'].values

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float)
X_test_tensor = torch.tensor(X_test, dtype=torch.float)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Define the MLP model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [18]:
# Model instantiation
input_size = X_train.shape[1]
hidden_size = 100  # Example hidden size
num_classes = len(torch.unique(y_train_tensor))
model = MLP(input_size, hidden_size, num_classes)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
num_epochs = 10
for epoch in range(num_epochs):
    # Training
    model.train()  # Set the model to train mode
    correct_train = 0
    total_train = 0
    for i, (features, labels) in enumerate(train_loader):
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate training accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

        if (i + 1) % 100 == 0:
            print(f'Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], Loss: {loss.item():.4f}')

    train_accuracy = 100 * correct_train / total_train
    print(f'Training Accuracy at epoch {epoch + 1}: {train_accuracy:.2f}%')

    # Evaluation on test set
    model.eval()  # Set the model to evaluation mode
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_accuracy = 100 * correct_test / total_test
    print(f'Test Accuracy at epoch {epoch + 1}: {test_accuracy:.2f}%')


Epoch [1/10], Step [100/12500], Loss: 0.9545
Epoch [1/10], Step [200/12500], Loss: 0.9065
Epoch [1/10], Step [300/12500], Loss: 0.8255
Epoch [1/10], Step [400/12500], Loss: 0.8220
Epoch [1/10], Step [500/12500], Loss: 0.4613
Epoch [1/10], Step [600/12500], Loss: 0.4050
Epoch [1/10], Step [700/12500], Loss: 0.3607
Epoch [1/10], Step [800/12500], Loss: 0.2784
Epoch [1/10], Step [900/12500], Loss: 0.2697
Epoch [1/10], Step [1000/12500], Loss: 0.5386
Epoch [1/10], Step [1100/12500], Loss: 0.1386
Epoch [1/10], Step [1200/12500], Loss: 0.1511
Epoch [1/10], Step [1300/12500], Loss: 0.1112
Epoch [1/10], Step [1400/12500], Loss: 0.3143
Epoch [1/10], Step [1500/12500], Loss: 0.2064
Epoch [1/10], Step [1600/12500], Loss: 0.1645
Epoch [1/10], Step [1700/12500], Loss: 0.2065
Epoch [1/10], Step [1800/12500], Loss: 0.2257
Epoch [1/10], Step [1900/12500], Loss: 0.1085
Epoch [1/10], Step [2000/12500], Loss: 0.1586
Epoch [1/10], Step [2100/12500], Loss: 0.1304
Epoch [1/10], Step [2200/12500], Loss: 0.09

In [19]:
torch.save(model.state_dict(), 'bangyamodel.pth')